# UrbanRelay AI — Data Pipeline & Modeling Notebook
Dataset: `urbanrelay_top10_cities_by_area_COMBINED.csv` (170 rows, 10 cities x ~17 areas each)

**Sections:**
1. Data Loading
2. Data Cleaning
3. Data Visualization (Plotly: Line, Bar, Scatter, Histogram, Violin, Gantt, Heatmap, 3D)
4. Feature Selection & Scaling
5. Train/Test Split
6. Model Building
7. Model Evaluation


## 0. Imports

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

pd.set_option("display.max_columns", None)


## 1. Data Loading
Load the combined UrbanRelay dataset. Update `FILE_PATH` to wherever you saved the CSV.

In [2]:
FILE_PATH = "urbanrelay_top10_cities_by_area_COMBINED.csv"

df = pd.read_csv(FILE_PATH)
print("Shape:", df.shape)
df.head()


Shape: (170, 21)


,area_id,city,state,area_name,total_road_length_km,num_intersections,avg_traffic_volume_per_hr,avg_traffic_speed_kmph,congestion_level,num_micro_hubs,total_hub_storage_capacity,avg_hub_utilization_pct,num_delivery_orders_monthly,avg_package_weight_kg,num_vehicles_fleet,most_common_vehicle_type,num_couriers,avg_courier_capacity_kg,avg_daily_demand_orders,num_curb_zones,total_curb_capacity_vehicles
0,mum_andheri,Mumbai,Maharashtra,Andheri,16.13,13,1946,18.5,High,9,733,56.6,702,2.90,83,Two-Wheeler,16,20.5,234,11,20
1,mum_bandra,Mumbai,Maharashtra,Bandra,16.20,6,1114,13.5,Severe,13,2316,45.8,298,1.42,17,Mini Van,55,16.4,99,0,0
2,mum_dadar,Mumbai,Maharashtra,Dadar,72.84,19,1696,21.0,High,16,3174,45.4,247,2.79,15,Three-Wheeler (E),32,9.0,82,4,6
3,mum_colaba,Mumbai,Maharashtra,Colaba,68.88,23,2084,21.2,High,11,1357,32.7,374,2.67,15,Two-Wheeler,10,18.8,124,4,11
4,mum_powai,Mumbai,Maharashtra,Powai,78.66,12,1200,25.8,Moderate,13,2058,80.8,508,3.11,43,Mini Van,57,21.3,169,4,10


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 170 entries, 0 to 169
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   area_id                       170 non-null    str    
 1   city                          170 non-null    str    
 2   state                         170 non-null    str    
 3   area_name                     170 non-null    str    
 4   total_road_length_km          170 non-null    float64
 5   num_intersections             170 non-null    int64  
 6   avg_traffic_volume_per_hr     170 non-null    int64  
 7   avg_traffic_speed_kmph        170 non-null    float64
 8   congestion_level              170 non-null    str    
 9   num_micro_hubs                170 non-null    int64  
 10  total_hub_storage_capacity    170 non-null    int64  
 11  avg_hub_utilization_pct       170 non-null    float64
 12  num_delivery_orders_monthly   170 non-null    int64  
 13  avg_package_weig

In [4]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
area_id,170,170,mum_andheri,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,170,10,Mumbai,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,170,9,Maharashtra,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
area_name,170,170,Andheri,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_road_length_km,170.0,NaN,NaN,NaN,40.620765,20.286384,9.95,24.7825,36.705,55.37,101.02
num_intersections,170.0,NaN,NaN,NaN,12.158824,5.979606,2.0,7.0,12.0,16.0,31.0
avg_traffic_volume_per_hr,170.0,NaN,NaN,NaN,1210.205882,542.306371,370.0,733.0,1124.0,1632.75,2869.0
avg_traffic_speed_kmph,170.0,NaN,NaN,NaN,23.849412,7.185276,9.5,18.325,24.35,29.575,40.0
congestion_level,170,4,Moderate,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
num_micro_hubs,170.0,NaN,NaN,NaN,7.652941,4.105505,1.0,4.0,7.5,10.0,19.0


## 2. Data Cleaning
Check for missing values, duplicates, inconsistent text, and outliers before using the data.

In [5]:
# Missing values per column
df.isna().sum()


area_id                         0
city                            0
state                           0
area_name                       0
total_road_length_km            0
num_intersections               0
avg_traffic_volume_per_hr       0
avg_traffic_speed_kmph          0
congestion_level                0
num_micro_hubs                  0
total_hub_storage_capacity      0
avg_hub_utilization_pct         0
num_delivery_orders_monthly     0
avg_package_weight_kg           0
num_vehicles_fleet              0
most_common_vehicle_type        0
num_couriers                    0
avg_courier_capacity_kg         0
avg_daily_demand_orders         0
num_curb_zones                  0
total_curb_capacity_vehicles    0
dtype: int64

In [6]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Duplicate area_id (should be unique)
print("Duplicate area_id:", df["area_id"].duplicated().sum())


Duplicate rows: 0
Duplicate area_id: 0


In [7]:
# Standardize text columns (strip whitespace, fix casing)
text_cols = ["city", "state", "area_name", "most_common_vehicle_type", "congestion_level"]
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

df["congestion_level"] = df["congestion_level"].str.title()
df["most_common_vehicle_type"] = df["most_common_vehicle_type"].str.strip()


In [8]:
# Outlier check on key numeric columns using IQR
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

def iqr_outlier_count(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_summary = {c: iqr_outlier_count(df[c]) for c in numeric_cols}
pd.Series(outlier_summary, name="outlier_count").sort_values(ascending=False)


total_hub_storage_capacity      3
num_vehicles_fleet              3
num_couriers                    2
total_curb_capacity_vehicles    2
num_curb_zones                  2
num_intersections               1
avg_daily_demand_orders         1
num_delivery_orders_monthly     1
avg_hub_utilization_pct         0
avg_traffic_speed_kmph          0
avg_traffic_volume_per_hr       0
total_road_length_km            0
num_micro_hubs                  0
avg_package_weight_kg           0
avg_courier_capacity_kg         0
Name: outlier_count, dtype: int64

In [9]:
# No missing values or duplicates expected in this generated dataset,
# but this cleaning block will catch and handle them if the file is regenerated
# or replaced with a version that has real-world messiness.
df = df.drop_duplicates(subset="area_id").reset_index(drop=True)
df = df.dropna(subset=["congestion_level"])  # target column must not be null
print("Final shape after cleaning:", df.shape)


Final shape after cleaning: (170, 21)


## 3. Data Visualization (Plotly)

### 3.1 Line Chart — Average traffic speed by city (aggregated across areas)

In [10]:
city_speed = df.groupby("city", as_index=False)["avg_traffic_speed_kmph"].mean()
city_speed = city_speed.sort_values("avg_traffic_speed_kmph", ascending=False)

fig = px.line(
    city_speed, x="city", y="avg_traffic_speed_kmph",
    markers=True, title="Average Traffic Speed by City"
)
fig.update_layout(xaxis_title="City", yaxis_title="Avg Speed (km/h)")
fig.show()


### 3.2 Bar Chart — Total monthly delivery orders by city

In [11]:
city_orders = df.groupby("city", as_index=False)["num_delivery_orders_monthly"].sum()
city_orders = city_orders.sort_values("num_delivery_orders_monthly", ascending=False)

fig = px.bar(
    city_orders, x="city", y="num_delivery_orders_monthly",
    color="city", title="Total Monthly Delivery Orders by City"
)
fig.update_layout(xaxis_title="City", yaxis_title="Delivery Orders / Month", showlegend=False)
fig.show()


### 3.3 Scatter Plot — Fleet size vs traffic volume, colored by congestion level

In [12]:
fig = px.scatter(
    df, x="num_vehicles_fleet", y="avg_traffic_volume_per_hr",
    color="congestion_level", size="num_delivery_orders_monthly",
    hover_data=["area_id", "city"],
    title="Fleet Size vs Traffic Volume (bubble size = monthly orders)"
)
fig.update_layout(xaxis_title="Vehicles in Fleet", yaxis_title="Avg Traffic Volume / hr")
fig.show()


### 3.4 Histogram — Distribution of average traffic speed

In [13]:
fig = px.histogram(
    df, x="avg_traffic_speed_kmph", nbins=20, color="congestion_level",
    title="Distribution of Average Traffic Speed (km/h)"
)
fig.update_layout(xaxis_title="Avg Speed (km/h)", yaxis_title="Number of Areas")
fig.show()


### 3.5 Violin Plot — Traffic speed spread by city

In [14]:
fig = px.violin(
    df, x="city", y="avg_traffic_speed_kmph", color="city",
    box=True, points="all",
    title="Traffic Speed Distribution by City (Violin Plot)"
)
fig.update_layout(xaxis_title="City", yaxis_title="Avg Speed (km/h)", showlegend=False)
fig.show()


### 3.6 Gantt-Style Chart — Estimated daily order-processing window (top 15 busiest areas)
Plotly's Gantt/timeline chart needs a start and end time. This dataset doesn't track real task
schedules, so we derive an **illustrative** processing window per area: assume each area's hub
starts at 08:00 and the window length scales with `avg_daily_demand_orders` (more orders = longer
processing window needed). This is a simulated schedule for visualization, not real logged data.

In [15]:
import datetime as dt

top15 = df.sort_values("avg_daily_demand_orders", ascending=False).head(15).copy()

base_start = dt.datetime(2026, 1, 1, 8, 0)
top15["start"] = base_start
# scale window length: 1 order ~ 3 minutes of processing, capped for readability
top15["duration_hr"] = (top15["avg_daily_demand_orders"] * 3 / 60).clip(upper=10)
top15["end"] = top15["start"] + pd.to_timedelta(top15["duration_hr"], unit="h")

fig = px.timeline(
    top15, x_start="start", x_end="end", y="area_id", color="city",
    title="Estimated Daily Order-Processing Window — Top 15 Busiest Areas (illustrative)"
)
fig.update_yaxes(autorange="reversed")
fig.show()


### 3.7 Heatmap — Correlation between numeric features

In [16]:
corr_cols = [
    "total_road_length_km", "num_intersections", "avg_traffic_volume_per_hr",
    "avg_traffic_speed_kmph", "num_micro_hubs", "total_hub_storage_capacity",
    "avg_hub_utilization_pct", "num_delivery_orders_monthly", "avg_package_weight_kg",
    "num_vehicles_fleet", "num_couriers", "avg_courier_capacity_kg",
    "avg_daily_demand_orders", "num_curb_zones", "total_curb_capacity_vehicles"
]
corr_matrix = df[corr_cols].corr()

fig = px.imshow(
    corr_matrix, text_auto=".2f", aspect="auto", color_continuous_scale="RdBu_r",
    title="Correlation Heatmap — Numeric Features"
)
fig.show()


### 3.8 3D Scatter Plot — Fleet size, traffic volume, and speed by city

In [17]:
fig = px.scatter_3d(
    df, x="num_vehicles_fleet", y="avg_traffic_volume_per_hr", z="avg_traffic_speed_kmph",
    color="city", hover_data=["area_id"],
    title="3D View: Fleet Size vs Traffic Volume vs Traffic Speed"
)
fig.show()


## 4. Feature Selection & Scaling
Target variable: **`congestion_level`** (Low / Moderate / High / Severe) — predicting congestion
supports the project's core goal of dynamic route optimization.

In [18]:
# Encode target
le = LabelEncoder()
df["congestion_level_encoded"] = le.fit_transform(df["congestion_level"])
print(dict(zip(le.classes_, le.transform(le.classes_))))


{'High': np.int64(0), 'Low': np.int64(1), 'Moderate': np.int64(2), 'Severe': np.int64(3)}


In [19]:
# Candidate numeric features (drop identifiers and the target itself)
feature_cols = [
    "total_road_length_km", "num_intersections", "avg_traffic_volume_per_hr",
    "num_micro_hubs", "total_hub_storage_capacity", "avg_hub_utilization_pct",
    "num_delivery_orders_monthly", "avg_package_weight_kg", "num_vehicles_fleet",
    "num_couriers", "avg_courier_capacity_kg", "avg_daily_demand_orders",
    "num_curb_zones", "total_curb_capacity_vehicles"
]

X = df[feature_cols]
y = df["congestion_level_encoded"]


In [20]:
# Select the K best features by ANOVA F-value
K = 8
selector = SelectKBest(score_func=f_classif, k=K)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()].tolist()
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)

print("Top feature scores:")
print(scores)
print("\nSelected features:", selected_features)


Top feature scores:
num_delivery_orders_monthly     5.412361
avg_daily_demand_orders         5.398201
total_road_length_km            5.370651
total_hub_storage_capacity      4.928389
avg_traffic_volume_per_hr       4.739916
num_intersections               4.425126
num_curb_zones                  3.750074
num_micro_hubs                  3.557552
num_vehicles_fleet              3.377589
avg_hub_utilization_pct         2.631048
total_curb_capacity_vehicles    2.140930
num_couriers                    0.963582
avg_package_weight_kg           0.788954
avg_courier_capacity_kg         0.254867
dtype: float64

Selected features: ['total_road_length_km', 'num_intersections', 'avg_traffic_volume_per_hr', 'num_micro_hubs', 'total_hub_storage_capacity', 'num_delivery_orders_monthly', 'avg_daily_demand_orders', 'num_curb_zones']


In [21]:
# Scale the selected features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X[selected_features])
X_scaled = pd.DataFrame(X_scaled, columns=selected_features)
X_scaled.head()


,total_road_length_km,num_intersections,avg_traffic_volume_per_hr,num_micro_hubs,total_hub_storage_capacity,num_delivery_orders_monthly,avg_daily_demand_orders,num_curb_zones
0,-1.210818,0.141090,1.360795,0.329080,-0.505131,2.121360,2.127090,1.746304
1,-1.207357,-1.033014,-0.177925,1.306260,1.875239,-0.278531,-0.278622,-1.496585
2,1.592912,1.147465,0.898439,2.039145,3.165420,-0.581488,-0.581564,-0.317352
3,1.397130,1.818381,1.616015,0.817670,0.433183,0.172933,0.166880,-0.317352
4,1.880651,-0.026639,-0.018875,1.306260,1.487282,0.968937,0.968784,-0.317352


## 5. Train/Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (136, 8)
Test shape: (34, 8)


## 6. Model Building
Two models for comparison: Logistic Regression (baseline, linear) and Random Forest (non-linear,
handles feature interactions better — usually stronger fit for this kind of tabular data).

In [23]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)


In [24]:
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)


## 7. Model Evaluation

In [25]:
def evaluate(name, y_true, y_pred):
    print(f"--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 3))
    print("Precision:", round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3))
    print("Recall   :", round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3))
    print("F1-score :", round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3))
    print()
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))

evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Random Forest", y_test, y_pred_rf)


--- Logistic Regression ---
Accuracy : 0.5
Precision: 0.442
Recall   : 0.5
F1-score : 0.469

              precision    recall  f1-score   support

        High       0.67      0.73      0.70        11
         Low       0.44      0.57      0.50         7
    Moderate       0.42      0.45      0.43        11
      Severe       0.00      0.00      0.00         5

    accuracy                           0.50        34
   macro avg       0.38      0.44      0.41        34
weighted avg       0.44      0.50      0.47        34

--- Random Forest ---
Accuracy : 0.353
Precision: 0.319
Recall   : 0.353
F1-score : 0.335

              precision    recall  f1-score   support

        High       0.42      0.45      0.43        11
         Low       0.50      0.57      0.53         7
    Moderate       0.25      0.27      0.26        11
      Severe       0.00      0.00      0.00         5

    accuracy                           0.35        34
   macro avg       0.29      0.32      0.31        34
w

In [26]:
# Confusion matrix (Random Forest) as a Plotly heatmap
cm = confusion_matrix(y_test, y_pred_rf)

fig = px.imshow(
    cm, text_auto=True, x=le.classes_, y=le.classes_,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Confusion Matrix — Random Forest"
)
fig.show()


In [27]:
# Feature importance from Random Forest
importance = pd.Series(rf_clf.feature_importances_, index=selected_features).sort_values(ascending=False)

fig = px.bar(
    importance, x=importance.values, y=importance.index, orientation="h",
    title="Feature Importance — Random Forest", labels={"x": "Importance", "y": "Feature"}
)
fig.show()
